In [1]:
import pandas as pd
from pathlib import Path
import numpy as np

proj_root = Path.cwd().parent
processed_path = proj_root / "data" / "processed" / "sdss_clean.csv"

df_pre_fe = pd.read_csv(processed_path)

u_g = df_pre_fe["u"] - df_pre_fe["g"]
print(u_g)
print(type(u_g))

0        0.14260
1        0.93954
2        0.15099
3        0.06253
4        0.37746
          ...   
99949    0.16267
99950    1.56913
99951    0.13805
99952    0.84224
99953    0.04838
Length: 99954, dtype: float64
<class 'pandas.core.series.Series'>


In [2]:
df_pre_fe["u_g"] = df_pre_fe["u"] - df_pre_fe["g"]
df_pre_fe["g_r"] = df_pre_fe["g"] - df_pre_fe["r"]
df_pre_fe["r_i"] = df_pre_fe["r"] - df_pre_fe["i"]
df_pre_fe["i_z"] = df_pre_fe["i"] - df_pre_fe["z"]

df_pre_fe.head()

,objID,specObjID,redshift,class,u,g,r,i,z,ra,dec,u_g,g_r,r_i,i_z
0,1237651251506446568,7670772665687494656,1.433770,QSO,20.78556,20.64296,20.08354,19.78511,19.81030,221.303424,60.277896,0.14260,0.55942,0.29843,-0.02519
1,1237655108917592122,8013191927121072128,-0.000459,STAR,17.09996,16.16042,15.89365,15.81271,15.78566,197.711223,59.925209,0.93954,0.26677,0.08094,0.02705
2,1237651821098827886,601354828572026880,0.833289,QSO,19.12911,18.97812,19.03414,19.25292,19.06842,216.028829,3.068298,0.15099,-0.05602,-0.21878,0.18450
3,1237651250939429598,10541823497646395392,1.198927,QSO,20.59022,20.52769,20.35182,20.33298,20.56437,113.479782,34.044699,0.06253,0.17587,0.01884,-0.23139
4,1237651251510771972,693587094055970816,0.396555,QSO,19.56240,19.18494,18.63504,18.38647,17.98028,235.643009,54.026770,0.37746,0.54990,0.24857,0.40619


In [3]:
class_mapping = {"STAR": 0, "QSO": 1}
df_pre_fe["target"] = df_pre_fe["class"].map(class_mapping)

if df_pre_fe["target"].isna().any():
    unmapped = df_pre_fe.loc[df_pre_fe["target"].isna(), "class"].unique()
    raise ValueError(f"Found unexpected classes that could not be mapped: {unmapped}")

df_pre_fe["target"] = df_pre_fe["target"].astype(int)
df_pre_fe.head()

,objID,specObjID,redshift,class,u,g,r,i,z,ra,dec,u_g,g_r,r_i,i_z,target
0,1237651251506446568,7670772665687494656,1.433770,QSO,20.78556,20.64296,20.08354,19.78511,19.81030,221.303424,60.277896,0.14260,0.55942,0.29843,-0.02519,1
1,1237655108917592122,8013191927121072128,-0.000459,STAR,17.09996,16.16042,15.89365,15.81271,15.78566,197.711223,59.925209,0.93954,0.26677,0.08094,0.02705,0
2,1237651821098827886,601354828572026880,0.833289,QSO,19.12911,18.97812,19.03414,19.25292,19.06842,216.028829,3.068298,0.15099,-0.05602,-0.21878,0.18450,1
3,1237651250939429598,10541823497646395392,1.198927,QSO,20.59022,20.52769,20.35182,20.33298,20.56437,113.479782,34.044699,0.06253,0.17587,0.01884,-0.23139,1
4,1237651251510771972,693587094055970816,0.396555,QSO,19.56240,19.18494,18.63504,18.38647,17.98028,235.643009,54.026770,0.37746,0.54990,0.24857,0.40619,1


In [4]:
df_fe = df_pre_fe.copy()

raw_bands = ["u", "g", "r", "i", "z"]
color_features = ["u_g", "g_r", "r_i", "i_z"]
feature_bands = raw_bands + color_features

metadata_cols = ["objID", "specObjID", "ra", "dec", "redshift"]
target_col = "target"

df_fe.head()

,objID,specObjID,redshift,class,u,g,r,i,z,ra,dec,u_g,g_r,r_i,i_z,target
0,1237651251506446568,7670772665687494656,1.433770,QSO,20.78556,20.64296,20.08354,19.78511,19.81030,221.303424,60.277896,0.14260,0.55942,0.29843,-0.02519,1
1,1237655108917592122,8013191927121072128,-0.000459,STAR,17.09996,16.16042,15.89365,15.81271,15.78566,197.711223,59.925209,0.93954,0.26677,0.08094,0.02705,0
2,1237651821098827886,601354828572026880,0.833289,QSO,19.12911,18.97812,19.03414,19.25292,19.06842,216.028829,3.068298,0.15099,-0.05602,-0.21878,0.18450,1
3,1237651250939429598,10541823497646395392,1.198927,QSO,20.59022,20.52769,20.35182,20.33298,20.56437,113.479782,34.044699,0.06253,0.17587,0.01884,-0.23139,1
4,1237651251510771972,693587094055970816,0.396555,QSO,19.56240,19.18494,18.63504,18.38647,17.98028,235.643009,54.026770,0.37746,0.54990,0.24857,0.40619,1


In [5]:
cols_to_check = feature_bands + [target_col]

nan_count = df_fe[cols_to_check].isna().sum().sum()
print(f"NaN values in model features + target: {nan_count}")

all_finite = np.isfinite(df_fe[cols_to_check].to_numpy()).all()
print(f"All values are finite: {all_finite}")

NaN values in model features + target: 0
All values are finite: True


In [6]:
feature_path = proj_root / "data" / "processed" / "sdss_features.csv"
export_cols = feature_bands + [target_col]
df_fe[export_cols].to_csv(feature_path, index=False)

In [10]:
df = pd.read_csv(feature_path)
print(df.groupby("target")[["u_g", "g_r", "r_i", "i_z"]].mean())

             u_g       g_r       r_i      i_z
target                                       
0       1.375142  0.530689  0.260776  0.12977
1       0.381457  0.307564  0.189151  0.12214
